# 01 – Deskriptive Analyse (Leitfaden Punkt 3)

Ziel: Grafik der Zeitreihe und Zerlegung in Trend, Saison und Rest. Die Funktionen liegen in `src/plots.py`, hier stehen nur Aufrufe und Interpretation.

Daten: Destatis 45212-0004, WZ08-4711-02, real (2021 = 100), 01/1994–08/2026.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import load_umsatz
from src.evaluation import train_test_split
from src import plots

s = load_umsatz(str(ROOT / "data/raw/45212-0004_de.csv"))
train, test = train_test_split(s)
print(f"{len(s)} Monate ({s.index[0]:%m/%Y}–{s.index[-1]:%m/%Y}), fehlende Werte: {s.isna().sum()}")
print(f"Training: {len(train)} Monate, Test: {len(test)} Monate")
s.describe().round(2)

## 1. Zeitreihe mit gleitendem Durchschnitt (SMA, k = 12)

Mit k = 12 umfasst jedes Fenster genau ein Jahr. Die Saison mittelt sich heraus, übrig bleibt der Trend (Vorlesung 2).

In [ ]:
fig = plots.plot_zeitreihe(s)
plots.speichern(fig, "01_zeitreihe")

In [ ]:
# SMA zum Monat t = Mittel der 12 Monate bis einschließlich t
sma = s.rolling(12).mean().dropna()
print(f"Start   ({sma.index[0]:%m/%Y}): {sma.iloc[0]:.1f}")
print(f"Ende    ({sma.index[-1]:%m/%Y}): {sma.iloc[-1]:.1f}")
print(f"Maximum ({sma.idxmax():%m/%Y}): {sma.max():.1f}")
print(f"Minimum ({sma.idxmin():%m/%Y}): {sma.min():.1f}")

# SMA im Dezember = Mittel des Kalenderjahres
jahresmittel = sma[sma.index.month == 12]
jahresmittel.index = jahresmittel.index.year
print()
print("SMA im Dezember je Jahr:")
print(jahresmittel.round(1).to_string())

**Interpretation** (Zahlen aus der Zelle darüber):

- Der SMA (Fenster endet im genannten Monat) ist nicht konstant: Minimum 85,5 (Fenster bis 03/1998), Maximum 102,2 (Fenster bis 03/2021). Start 86,5 (12/1994), Ende 94,6 (08/2026).
- SMA im Dezember: steigt von 85,9 (1997) auf 95,0 (2005), fällt auf 86,0 (2009), liegt bis 2014 zwischen 86,0 und 88,1, steigt auf 101,4 (2020), fällt auf 92,4 (2023) und liegt 2025 bei 94,8.
- Der Mittelwert hängt vom Zeitfenster ab; das spricht gegen Stationarität (formale Tests: Person B).

## 2. Saisonfigur je Kalendermonat

In [ ]:
fig = plots.plot_saisonfigur(s)
plots.speichern(fig, "02_saisonfigur")
s.groupby(s.index.month).mean().round(1)

**Interpretation:**

- Dezember liegt im Mittel bei 105,6, Januar bei 83,8 und Februar bei 82,3 – die tiefsten Monate.
- März bis November liegen zwischen 87,3 (September) und 93,2 (März).

## 3. Lag-Plots (Vorlesung 1)

In [ ]:
fig = plots.plot_lags(s, lags=(1, 12))
plots.speichern(fig, "03_lagplots")
for k in (1, 12):
    print(f"Lag {k:2d}: r = {s.autocorr(k):.2f}")

**Interpretation:** Lag 1: r = 0,26 (schwacher Zusammenhang), Lag 12: r = 0,89 (starker Zusammenhang). Ein Monat ähnelt dem Vorjahresmonat deutlich stärker als dem Vormonat. Das spricht für eine saisonale Differenz (Lag 12) im ARIMA-Teil; Seasonal Naive nutzt genau diesen Zusammenhang (Prognose = Wert des Vorjahresmonats).

## 4. STL-Zerlegung

STL zerlegt y_t = T_t + S_t + R_t. In der Vorlesung wird STL nur zur **Beschreibung** genutzt, nicht zur Prognose. `robust=True`, damit Ausreißer im Rest landen statt Trend und Saison zu verzerren.

In [ ]:
res = plots.stl_zerlegung(s)
fig = plots.plot_stl(res)
plots.speichern(fig, "04_stl")

In [ ]:
# Saisonamplitude (max − min der Saisonkomponente) je vollständigem Jahr, gesamte Reihe
amp = plots.saisonamplitude(res)["amplitude"]
print(f"Erstes volles Jahr {amp.index[0]}: {amp.iloc[0]:.2f}")
print(f"Letztes volles Jahr {amp.index[-1]}: {amp.iloc[-1]:.2f}")
print(f"Maximum {amp.idxmax()}: {amp.max():.2f}")
print(f"Minimum {amp.idxmin()}: {amp.min():.2f}")
print(f"Trend Start   ({res.trend.index[0]:%m/%Y}): {res.trend.iloc[0]:.1f}")
print(f"Trend Maximum ({res.trend.idxmax():%m/%Y}): {res.trend.max():.1f}")
print(f"Trend Ende    ({res.trend.index[-1]:%m/%Y}): {res.trend.iloc[-1]:.1f}")
print(f"2023 (gesamte Reihe): {amp[2023]:.2f}")

In [ ]:
# Größte Reste: welche Monate weichen am stärksten vom Muster ab?
print(f"Mittelwert Rest: {res.resid.mean():.2f}")
top = res.resid.reindex(res.resid.abs().sort_values(ascending=False).index[:8])
tabelle = top.round(2).to_frame("rest")
tabelle["monat"] = [plots.MONATSNAMEN[m - 1] for m in top.index.month]
tabelle["vorzeichen"] = top.gt(0).map({True: "+", False: "−"})
maerz_april = top.index.month.isin([3, 4])
print(f"Davon im März/April: {maerz_april.sum()} von {len(top)} "
      f"({top[maerz_april].gt(0).sum()} positiv, {top[maerz_april].lt(0).sum()} negativ)")
print(f"Erwartet bei Gleichverteilung über 12 Monate: {len(top) * 2 / 12:.1f}")
tabelle

**Interpretation:**

- **Trend:** Start 84,7 (01/1994), Maximum 101,0 (10/2020), Ende 94,6 (08/2026).
- **Saison:** über die Zeit **nicht konstant**. Die Amplitude liegt 1994 bei 23,95, erreicht 2003 das Maximum von 26,83 und liegt 2025 beim Minimum von 17,05. Die Werte der letzten Jahre hängen vom Ende der Reihe ab: für 2023 ergibt die STL 18,34 (gesamte Reihe) bzw. 17,70 (nur Training, Abschnitt 5).
- **Rest:** Mittelwert 0,03. Von den 8 größten absoluten Resten liegen 5 im März oder April (3 positiv, 2 negativ); bei Gleichverteilung über 12 Monate wären 1,3 zu erwarten (Orientierung, kein Test). Ob ein Kalendereffekt die Ursache ist, wurde nicht geprüft.

## 5. Additiv oder multiplikativ? (nur Trainingsdaten)

Wächst die Saisonamplitude mit dem Niveau, ist ein multiplikatives Modell bzw. eine Box-Cox-Transformation sinnvoll. Die Entscheidung fällt auf den Trainingsdaten, weil sie die Modellwahl beeinflusst.

In [ ]:
tab = plots.saisonamplitude(plots.stl_zerlegung(train))
fig = plots.plot_saisonamplitude(tab)
plots.speichern(fig, "05_saisonamplitude")
print("Korrelation Amplitude–Niveau:", round(tab["amplitude"].corr(tab["niveau"]), 2))
tab.round(2)

**Entscheidung:** Die Korrelation zwischen Saisonamplitude und Niveau liegt auf den Trainingsdaten bei r = −0,23, also kein positiver Zusammenhang. → **additive** Saison für Holt-Winters. Ob für ARIMA eine Box-Cox-Transformation nötig ist, prüft Person B.

## Fazit für die Modellierung

1. Der SMA liegt zwischen 85,5 und 102,2 (Mittelwert nicht konstant) → voraussichtlich Differenzen (saisonal und ggf. regulär) für ARIMA; Entscheidung nach den Stationaritätstests (Person B).
2. Saison mit Periode 12: Lag-12-Korrelation r = 0,89 (Lag 1: r = 0,26).
3. Additive Saison: Korrelation Amplitude–Niveau r = −0,23 (Training).
4. Das Niveau ändert sich deutlich über die Zeit (SMA zwischen 85,5 und 102,2). Die Saisonamplitude liegt 1994 bei 23,95, 2003 bei 26,83 (Maximum) und 2025 bei 17,05 (Minimum) → Einschränkungen für die Diskussion.